# 01 - Exploratory analysis
Run the notebook from the project root after `python src/data/prepare.py` and `python src/features/build_rfm.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

clean = pd.read_parquet("data/processed/clean.parquet")
rfm = pd.read_parquet("data/features/rfm.parquet")
print(clean.shape, rfm.shape)
clean.head()

## Transactions

In [ ]:
print(clean["invoice_date"].min(), "->", clean["invoice_date"].max())
print("customers:", clean["customer_id"].nunique(), " invoices:", clean["invoice"].nunique())
monthly = clean.set_index("invoice_date")["revenue"].resample("MS").sum()
monthly.plot(title="Monthly revenue", figsize=(9, 3)); plt.show()

## RFM distributions (heavily right-skewed -> we log-transform before clustering)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12, 6))
for i, c in enumerate(["recency", "frequency", "monetary"]):
    rfm[c].plot.hist(bins=50, ax=ax[0, i], title=c)
    np.log1p(rfm[c]).plot.hist(bins=50, ax=ax[1, i], title=f"log1p({c})")
plt.tight_layout(); plt.show()
rfm[["recency", "frequency", "monetary"]].describe().round(2)

## Correlation

In [ ]:
np.log1p(rfm[["recency", "frequency", "monetary"]]).corr().round(2)

## Quick look at k (elbow + silhouette) - the real search is in `src/models/tune.py`

In [ ]:
X = StandardScaler().fit_transform(np.log1p(rfm[["recency", "frequency", "monetary"]]))
ks, inertia, sil = range(2, 11), [], []
for k in ks:
    km = KMeans(k, n_init=10, random_state=42).fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_, sample_size=min(5000, len(X)), random_state=42))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(list(ks), inertia, marker="o"); ax[0].set_title("Elbow (inertia)")
ax[1].plot(list(ks), sil, marker="o"); ax[1].set_title("Silhouette")
plt.show()